# Usage (editable working copy of `docs/source/usage.rst`)

Edit the Markdown cells (prose/headings) and Python code cells
(examples) below. The Python cells are live -- run them to confirm an
edited example still behaves as commented.

When you're done, **File → Download as → reStructuredText (.rst)**
(or `jupyter nbconvert --to rst usage.ipynb`) gives you a `.rst`
starting point. `nbconvert`'s Markdown→RST conversion is close but
not perfect for Sphinx specifics (cross-references like
`` :doc:`api` ``, the exact `.. code-block:: python` directive
formatting, etc.) -- compare the result against the current
`usage.rst` and touch up those bits by hand before committing.


In [1]:
>>> from hyprat import Hy
>>> from IPython.display import display, Math  # for LaTeX output

# Usage

## Constructing Hypercomplex Numbers, Recursively, via the Class `Hy`

The `Hy` class represents a "tower" of multi-dimensional numbers, and every `Hy` has only two components: `real` and `imag`.

At the lowest level of the tower, the two components are rational numbers, represented by the `fractions.Fraction` class.

Every instance of a `Hy` has a property called `rank`, which will be a non-negative integer (rank = 0, 1, 2, ...). Although the `Fraction` class is different than the `Hy` class, for consistency it is considered here to have rank 0.

If a `Hy` has rank $n$, where $n=1, 2, \dots$, then its two components, `real` and `imag`, will both always have rank $n-1$. The `Hy` constructor will normalize whatever it is given so this invariant holds. That is, if `real` has rank $m$ and `imag` has rank $n$, and $m \ne n$, then the constructor will coerce the lower rank `Hy` into an instance of the higher rank `Hy` before constructing the new, higher rank `Hy`.

So, starting at the lowest level of the tower, as noted above, a `Hy` made up of two rational numbers represents a **rational complex number** and has rank 1, as shown in the following table.

| Hypercomplex | Rank | Dimension |
| ---: | :---: | :---: |
| Rational | 0 | $1 = 2^0$ |
| Complex     | 1 | $2 = 2^1$ |
| Quaternion | 2 | $4 = 2^2$ |
| Octonion | 3 | $8 = 2^3$ |
| in general | n | $d = 2^n$ |

### Rational Complex Numbers

Combine two rational numbers (`Fraction`s) to create a **rational complex number**. Floats, ints, and strings can be used for the two rational numbers, and they can be mixed:

In [2]:
>>> z1 = Hy("2/3", 1.5)
>>> print(f"{z1 = } has rank {z1.rank}\n")

>>> print(f"{str(z1) = }")

z1 = Hy('2/3', '3/2') has rank 1

str(z1) = '(2/3+3/2j)'


### Rational Quaternions

Combine two rational complex numbers (`Hy`s of rank 1) to create a **rational quaternion** (rank 2):

In [3]:
>>> z2 = Hy(4, "-1/7")  # another rank 1 Hy (complex number)

>>> q1 = Hy(z1, z2)  # rational quaternion
>>> print(f"{q1 = } has rank {q1.rank}\n")

>>> print(f"{str(q1) = }")

q1 = Hy(Hy('2/3', '3/2'), Hy('4', '-1/7')) has rank 2

str(q1) = '(2/3+3/2i+4j-1/7k)'


### Rational Octonions

Combine two rational quaternions (`Hy`s of rank 2) to create a **rational octonion** (rank 3):

In [4]:
>>> q2 = Hy(Hy('4/3', '-9/4'), Hy('-6/5', '2/5'))  # another rank 2 Hy (quaternion)

>>> o1 = Hy(q1, q2)  # rational octonion
>>> print(f"{o1 = } has rank {o1.rank}\n")

>>> print(f"{str(o1) = }")

o1 = Hy(Hy(Hy('2/3', '3/2'), Hy('4', '-1/7')), Hy(Hy('4/3', '-9/4'), Hy('-6/5', '2/5'))) has rank 3

str(o1) = '(2/3+3/2i+4j-1/7k+4/3L-9/4iL-6/5jL+2/5kL)'


## Other Ways to Construct Hypercomplex Numbers

There are three more ways to build a Hy:

* from a flat list of coefficients (`to_array` and `Hy.from_array`)
* from a string representation (`Hy.parse`)
* randomly generated (`Hy.random`)

Examples follow:

### From a Flat List of Coefficients

`Hy`s can be converted both **to** and **from** flat lists of coefficients.

**To a Flat List**:

In [25]:
>>> q1_coef = q1.to_array(as_str=True)  # setting as_str to False (default) returns Fractions
>>> q1_coef

['2/3', '3/2', '4', '-1/7']

**From a Flat List**:

In [26]:
>>> q1_copy = Hy.from_array(q1_coef)
>>> q1_copy == q1

True

### From a String Representation

In [27]:
>>> o1_str = str(o1)

>>> o1_copy = Hy.parse(o1_str)
>>> o1_copy == o1

True

### From a Random Number Generator (RNG)

In [28]:
>>> print(f"   complex: {Hy.random(1)}")
>>> print(f"quaternion: {Hy.random(2)}")
>>> print(f"  octonion: {Hy.random(3)}")
>>> print(f"  sedenion: {Hy.random(4)}")

   complex: (5/2j)
quaternion: (3-3/2i+2/3j+5/4k)
  octonion: (-7/4+9/5i+6j+4/5k+3/2L+7/3iL+4/5jL+6/5kL)
  sedenion: (7+9/2e1-5e2+7/6e3-e4+e5-5/4e6-2e7+5/6e8+e9+8/3e10+2e11+7/6e13+3/2e14+2e15)


There's more below on random generation of hypercomplex numbers.

## Arithmetic

`+`, `-`, `*` and `/` are all defined recursively via the Cayley-Dickson construction, so they work uniformly at every rank, with the exception of division at ranks $\ge 4$ (i.e., sedenions, pathions, ...).


In [10]:
print(f"{z1 + z2 = }")
print(f"{z1 - z2 = }")
print(f"{z1 * z2 = }")
print(f"{z1 / z2 = }")

z1 + z2 = Hy('14/3', '19/14')
z1 - z2 = Hy('-10/3', '23/14')
z1 * z2 = Hy('121/42', '124/21')
z1 / z2 = Hy('721/4710', '896/2355')


Multiplication is non-commutative for quaternions and octonions, and
non-associative for octonions, exactly as it should be:


In [11]:
i = Hy(Hy(0, 1), Hy(0, 0))
j = Hy(Hy(0, 0), Hy(1, 0))
print(f"{str(i * j) = }")
print(f"{str(j * i) = }")

str(i * j) = '(k)'
str(j * i) = '(-k)'


## More on Random Hypercomplex Numbers

`Hy.random(rank)`, where `rank` is a positive integer, draws a random rank-`rank` value: each of its `2**rank` coefficients is an independent `Fraction(n, d)`, with `n` uniform over `[lo, hi]` (default `[-9, 9]`) and `d` uniform over `[1, dmax]` (default `[1, 6]`). The defaults for `lo`, 'hi`, and `dmax` are set to small values to make examples, demos, and tests easy to read.

There are a few ways to control reproducibility of random output.

In [12]:
# A one-off seed, scoped to just a single call:
Hy.random(2, seed=7)

# Hy.seed(...) sets a shared default RNG for everything that
# follows, so plain Hy.random(rank) calls become reproducible also:
Hy.seed(2026)
Hy.random(2)

Hy(Hy('-2', '7/5'), Hy('-3', '2'))

In [13]:
# Or use your own random.Random for full control:
import random
Hy.random(2, rng=random.Random(123))

Hy(Hy('-8/3', '-7/4'), Hy('-1', '-2'))

## Units

`Hy.units(rank)` returns a dictionary of all units for the particular, `rank`, where the keys are the string representation of the units and the values are the `Hy`s.

In [14]:
Hy.units(1)

{'1': Hy('1', '0'),
 '-1': Hy('-1', '0'),
 'j': Hy('0', '1'),
 '-j': Hy('0', '-1')}

In [15]:
Hy.units(2).keys()

dict_keys(['1', '-1', 'i', '-i', 'j', '-j', 'k', '-k'])

`rank == 0` is the one case where the "hypercomplex value" in question is actually a `Fraction` rather than a `Hy`, so `Hy.units(0)` returns `{'1': Fraction(1, 1), '-1': Fraction(-1, 1)}`.

In [16]:
Hy.units(0)

{'1': Fraction(1, 1), '-1': Fraction(-1, 1)}

`some_hy.is_unit()` answers the corresponding membership question
for a single value, without building the whole dict:


In [17]:
Hy(0, 1).is_unit()   # j is a unit

True

In [18]:
Hy(1, 1).is_unit()   # not a unit

False

## LaTeX rendering

`some_hy.latex()` renders a value as a LaTeX math expression, which is useful in a Jupyter notebook.

Two keyword-only options are available:

* `vinculum` controls how a non-integer coefficient's fraction bar
  is typeset: `"horizontal"` (the default) uses `\frac{num}{den}`;
  `"diagonal"` uses a plain slash, `num/den`:
* `mode` controls whether the result is wrapped in LaTeX math
  delimiters: `"plain"` (the default) returns the bare expression,
  `"inline"` wraps it in `$...$`, and `"display"` wraps it in
  `\[...\]`.

Basis units render the same way [the API reference](api.rst)
describes for `str()`: `j` at rank 1, `i`/`j`/`k` at rank
2, and `i`/`j`/`k`/`L`/`iL`/`jL`/`kL` at rank 3, all
unchanged, except that the `e1, e2, ...` labels used from rank 4
up are subscripted (rendered as `e_{1}`, `e_{2}`, ...).

See [api](api.rst) for the full reference.

In [19]:
Hy('5/2', '-16/5').latex()

'\\frac{5}{2}-\\frac{16}{5}j'

In [20]:
Hy('5/2', '-16/5').latex(vinculum='diagonal')

'5/2-16/5j'

The following shows how to display `Hy`s in a Jupyter notebook:

In [21]:
from IPython.display import display, Math

print(f"{z1 = }")
display(Math(z1.latex()))

z1 = Hy('2/3', '3/2')


<IPython.core.display.Math object>

In [22]:
print(f"{q1 = }")
display(Math(q1.latex()))

q1 = Hy(Hy('2/3', '3/2'), Hy('4', '-1/7'))


<IPython.core.display.Math object>

In [23]:
print(f"{o1 = }")
display(Math(o1.latex()))

o1 = Hy(Hy(Hy('2/3', '3/2'), Hy('4', '-1/7')), Hy(Hy('4/3', '-9/4'), Hy('-6/5', '2/5')))


<IPython.core.display.Math object>

In [24]:
s1 = Hy.random(4)  # a random sedenion
print(f"{s1 = }")
display(Math(s1.latex()))

s1 = Hy(Hy(Hy(Hy('9/5', '6/5'), Hy('5/2', '-9/5')), Hy(Hy('-7', '0'), Hy('5', '1'))), Hy(Hy(Hy('1/2', '1'), Hy('2/3', '1/2')), Hy(Hy('7/6', '-7/6'), Hy('1', '8/5'))))


<IPython.core.display.Math object>

---
**RST-specific notes for when you convert this back:**
- The `[the API reference](api.rst)` / `[api](api.rst)` links above
  should become Sphinx cross-references, `` :doc:`the API reference <api>` ``
  and `` :doc:`api` ``, in the final `.rst` -- Markdown links don't
  carry that role automatically.
- Section headers here use `#`/`##`; in the original `.rst` these are
  underlined (`=====`, `-----`) rather than `#`-prefixed.